In [6]:
import pandas as pd

df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

print(df.shape)

print(df.columns)

display(df.head())

(5672, 12)
Index(['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물',
       '출판사', '발행(출시)일자', '분야'],
      dtype='str')


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설
3,9791175772892,S000220307771,테오,"21,000","18,900",10%,5%,"1,050",앨런 레비,오팬하우스,20260701,소설
4,9791198547514,S000211748790,수족관,"17,700","15,930",10%,5%,880,유래혁,포스터샵,20240111,소설


In [7]:

df["분야"].value_counts()

print(df.isna().sum())

print("중복 상품명:", df["상품명"].duplicated().sum())


상품코드        0
판매상품 ID     0
상품명         0
정가          0
판매가         0
할인율         0
적립율         0
적립예정포인트     0
인물          0
출판사         0
발행(출시)일자    0
분야          0
dtype: int64
중복 상품명: 71


### 실습 2. Baseline을 먼저 측정하기

In [8]:
# 실습 2. Baseline을 먼저 측정하기

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 1. 입력 데이터(X)와 정답(y)
X = df["상품명"]
y = df["분야"]

# 2. Train / Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 3. 기본 TF-IDF
baseline_vectorizer = TfidfVectorizer()

# 훈련 데이터에만 fit
X_train_vec = baseline_vectorizer.fit_transform(X_train)

# 테스트 데이터는 transform만
X_test_vec = baseline_vectorizer.transform(X_test)

# 4. Multinomial Naive Bayes 모델
baseline_model = MultinomialNB()

baseline_model.fit(X_train_vec, y_train)

# 5. 예측
baseline_pred = baseline_model.predict(X_test_vec)

# 6. 평가
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_macro_f1 = f1_score(
    y_test,
    baseline_pred,
    average="macro"
)

correct = (baseline_pred == y_test).sum()
total = len(y_test)

print("Baseline Accuracy:", baseline_accuracy)
print("Macro F1:", baseline_macro_f1)
print(f"정답 = {correct} / {total}")

print("\n분류 결과")
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 0.5374449339207048
Macro F1: 0.3884875504731808
정답 = 610 / 1135

분류 결과
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       0.80      0.08      0.14        53
       경제/경영       0.58      0.62      0.60       200
          소설       0.40      0.78      0.53       200
       시/에세이       0.56      0.57      0.57       200
          요리       1.00      0.05      0.10        38
          인문       0.52      0.49      0.51       200
   취미/실용/스포츠       1.00      0.15      0.27        72
      컴퓨터/IT       0.80      0.78      0.79       129

    accuracy                           0.54      1135
   macro avg       0.63      0.39      0.39      1135
weighted avg       0.59      0.54      0.50      1135



c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\llm-data-analysis-course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

### 실습 3. 한국어 형태소 분석 적용하기

In [9]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

sample = "처음 배우는 파이썬 데이터 분석"

for token in kiwi.tokenize(sample):
    print(token.form, token.tag)

처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG


In [10]:
# 사용할 품사
TARGET_TAGS = {"NNG", "NNP", "SL"}

def tokenize_title(text):
    tokens = kiwi.tokenize(str(text))

    words = [
        token.form
        for token in tokens
        if token.tag in TARGET_TAGS
    ]

    return " ".join(words)

In [11]:
df["상품명_토큰"] = df["상품명"].apply(tokenize_title)

df[["상품명", "상품명_토큰"]].head(10)

,상품명,상품명_토큰
0,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항
3,테오,테오
4,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자
6,모순,모순
7,서리꽃 세트,서리 꽃 세트
8,데미안,데미안
9,브람스를 좋아하세요,브람스


### 실습 4. 불필요한 단어 필터링하기

In [12]:
# 불용어
stopwords = {
    "에디션",
}

def clean_tokens(text):
    words = str(text).split()

    cleaned = [
        word
        for word in words
        if len(word) >= 2          # 너무 짧은 토큰 제거
        and not word.isdigit()     # 숫자로만 된 토큰 제거
        and word not in stopwords  # 불용어 제거
    ]

    return " ".join(cleaned)

df["상품명_정제"] = df["상품명_토큰"].apply(clean_tokens)

display(
    df[["상품명", "상품명_토큰", "상품명_정제"]].head(10)
)

,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스


### 실습 5. Improved 분류 모델 평가하기

In [13]:
# 실습 5. Improved 분류 모델 평가하기

# 실습 3 + 4에서 만든 함수를 이용해 전처리
train_texts = X_train.apply(tokenize_title).apply(clean_tokens)
test_texts = X_test.apply(tokenize_title).apply(clean_tokens)

print("훈련 데이터:", len(train_texts))
print("테스트 데이터:", len(test_texts))

display(train_texts.head())

훈련 데이터: 4537
테스트 데이터: 1135


3717    The 모두의 스도쿠 No
4233      하루 이해 반도체 산업
596              화산 귀환
2416                고전
2221        미움받을 용기 특별
Name: 상품명, dtype: str

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Improved TF-IDF
improved_vectorizer = TfidfVectorizer()

# 훈련 데이터에만 fit
X_train_improved = improved_vectorizer.fit_transform(train_texts)

# 테스트 데이터는 transform만
X_test_improved = improved_vectorizer.transform(test_texts)

# Improved 모델
improved_model = MultinomialNB()

improved_model.fit(X_train_improved, y_train)

# 예측
improved_pred = improved_model.predict(X_test_improved)

In [15]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

improved_accuracy = accuracy_score(y_test, improved_pred)

improved_macro_f1 = f1_score(
    y_test,
    improved_pred,
    average="macro"
)

improved_correct = (improved_pred == y_test).sum()
total = len(y_test)

print("Improved Accuracy:", improved_accuracy)
print("Improved Macro F1:", improved_macro_f1)
print(f"정답 = {improved_correct} / {total}")

print("\n분류 결과")
print(classification_report(y_test, improved_pred))

Improved Accuracy: 0.5788546255506608
Improved Macro F1: 0.49098332880257584
정답 = 657 / 1135

분류 결과
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135



In [16]:
print("=== Before / After ===")

print(
    f"Baseline : Accuracy {baseline_accuracy:.4f} "
    f"/ Macro F1 {baseline_macro_f1:.4f} "
    f"/ 정답 {(baseline_pred == y_test).sum()} / {len(y_test)}"
)

print(
    f"Improved : Accuracy {improved_accuracy:.4f} "
    f"/ Macro F1 {improved_macro_f1:.4f} "
    f"/ 정답 {improved_correct} / {len(y_test)}"
)

print(
    f"\nAccuracy 변화: "
    f"{(improved_accuracy - baseline_accuracy) * 100:+.2f}%p"
)

print(
    f"정답 수 변화: "
    f"{improved_correct - (baseline_pred == y_test).sum():+d}권"
)

=== Before / After ===
Baseline : Accuracy 0.5374 / Macro F1 0.3885 / 정답 610 / 1135
Improved : Accuracy 0.5789 / Macro F1 0.4910 / 정답 657 / 1135

Accuracy 변화: +4.14%p
정답 수 변화: +47권


### 실습 6. 추천 결과의 문제 개선하기

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recommend_books(selected_index, top_n=5):
    # 1. 선택한 도서
    selected_book = df.loc[selected_index]
    selected_category = selected_book["분야"]

    print("선택 도서:", selected_book["상품명"])
    print("분야:", selected_category)

    # 2. 같은 분야의 도서만 후보로 선택
    candidate_df = df[
        df["분야"] == selected_category
    ].copy()

    # 선택한 도서도 포함되어 있어야 TF-IDF 비교 가능
    # 3. 형태소 분석 + 단어 필터링
    candidate_df["추천_텍스트"] = (
        candidate_df["상품명"]
        .apply(tokenize_title)
        .apply(clean_tokens)
    )

    # 4. TF-IDF
    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(
        candidate_df["추천_텍스트"]
    )

    # 선택 도서가 candidate_df에서 몇 번째인지 찾기
    selected_position = candidate_df.index.get_loc(selected_index)

    # 5. Cosine Similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_position],
        tfidf_matrix
    ).flatten()

    candidate_df["similarity"] = similarity_scores

    # 6. 자기 자신 제외 + 유사도 0 제외
    recommendations = candidate_df[
        (candidate_df.index != selected_index)
        & (candidate_df["similarity"] > 0)
    ]

    # 7. 유사도 높은 순으로 정렬
    recommendations = recommendations.sort_values(
        "similarity",
        ascending=False
    ).head(top_n)

    # 8. 추천 결과
    if len(recommendations) == 0:
        print(
            "\n현재 기준으로 유사도가 있는 "
            "추천 도서를 찾지 못했습니다."
        )
        return recommendations

    return recommendations[
        ["상품명", "분야", "similarity"]
    ]

result = recommend_books(0)

display(result)

df[df["상품명"].str.contains("거짓말이 달려온다", na=False)]

result = recommend_books(123)

display(result)

선택 도서: 싯다르타
분야: 소설


,상품명,분야,similarity
334,싯다르타,소설,1.000000
550,싯다르타,소설,1.000000
801,싯다르타,소설,1.000000
581,싯다르타(완역본),소설,0.635560
515,이지페이지: 싯다르타(큰글자책),소설,0.517212


선택 도서: 궤도
분야: 소설

현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야,상품명_토큰,상품명_정제,추천_텍스트,similarity


### 실습 7. Streamlit 앱에서 최종 검증하기